<a id="top"></a>
# Estimating Galaxy Cluster Masses with Convolutional Neural Networks

***

## Imports
This notebook uses the following:
- *numpy* to handle array functions
- *astropy.io fits* for accessing FITS files
- *matplotlib.pyplot* for plotting data
- *keras* for building the CNN

In [ ]:
%matplotlib widget
import numpy as np
from astropy.io import fits
import matplotlib.pyplot as plt
import pandas as pd
import keras
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from keras.models import Sequential
from keras.layers import Input, Dense, Dropout, Conv2D, MaxPooling2D
from keras.layers import GlobalAveragePooling2D


# Approch 1

## Cluster Mass Regression

This notebook trains a CNN to estimate cluster mass from X-ray images. The key evaluation unit is the cluster, not an individual image, because each cluster has up to three lines of sight.

### 1. Load the data

The FITS file contains the X-ray images and metadata, including `cluster_id`, `view_los`, `log_M500`, and the original train/validation/test flags.

In [ ]:
filename = r'/Users/tripathimihir/Documents/a3net_2026/Cluster Mass Hack/cluster_TNG_data.fits'
hdul = fits.open(filename)
image_size = hdul[1].data.shape[1]

### 2. Prepare the baseline split

The original split is retained for the baseline model. The cross-validation model below uses cluster-level folds instead, preventing different views of one cluster from appearing in both training and validation data.

In [ ]:
train_ind = np.argwhere(hdul[2].data['train'] == 1)
train_X = hdul[1].data[train_ind].reshape(-1, image_size, image_size, 1)
train_Y = hdul[2].data['log_M500'][train_ind]

val_ind = np.argwhere(hdul[2].data['validate'] == 1)
val_X = hdul[1].data[val_ind].reshape(-1, image_size, image_size, 1)
val_Y = hdul[2].data['log_M500'][val_ind]

test_ind = np.argwhere(hdul[2].data['test'] == 1)
test_X = hdul[1].data[test_ind].reshape(-1, image_size, image_size, 1)
test_Y = hdul[2].data['log_M500'][test_ind]

ML models often converge faster when the output is _roughly_ between 0 and 1. Next, let's perform a normalization that will speed up the training process a bit -- calculate the minimum log(mass) and subtract it from the Y labels as a change of units.  We'll put it back, later in the notebook, just before we plot the results.  

We can simply subtract off a constant value because our output doesn't have a large dynamical range, but there may be cases where you want to handle this more carefully and force all of your labels to be strictly in the (0,1) interval.  

In [ ]:
norm = np.nanmin(hdul[2].data['log_M500'])
train_Y -= norm
val_Y -= norm
test_Y -= norm

### 3. Build a CNN in Keras

Here, we will build the model described in Section 2.2 of [Ntampaka et al., 2019](https://ui.adsabs.harvard.edu/abs/2019ApJ...876...82N/abstract)

Further details about Conv2D, MaxPooling2D, GlobalAveragePooling2D, Dropout, and Dense layers can be found in the [Keras Layers Documentation](https://keras.io/api/layers/).  Further details about the relu activation function can be found in the [Keras Activation Function Documentation](https://keras.io/api/layers/activations/).

In [ ]:
input_shape = (image_size, image_size, 1)
model = Sequential([
    Input(shape=input_shape),
    Conv2D(16, kernel_size=(3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    Conv2D(32, kernel_size=(3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    Conv2D(64, kernel_size=(3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    GlobalAveragePooling2D(),
    Dropout(0.1),
    Dense(200, activation='relu'),
    Dropout(0.1),
    Dense(100, activation='relu'),
    Dense(20, activation='relu'),
    Dense(1, activation='linear')
])

### 4. Compile the CNN

Next, we compile the model.  As in [Ntampaka et al., 2019](https://ui.adsabs.harvard.edu/abs/2019ApJ...876...82N/abstract), we select the Adam opmimizer (with a learning rate that is slightly decreased from the default rate) and the mean squared error loss function.

You can learn more about [optimizers](https://keras.io/api/optimizers/) and more about [loss functions for regression tasks](https://keras.io/api/losses/) in the [Keras documentation](https://keras.io/)

In [ ]:
model.compile(loss='mean_squared_error', optimizer=keras.optimizers.Adam(learning_rate=0.0002))

### 5. Train the CNN to perform a regression task

We will start with training for 5 epochs, but this almost certainly won't be long enough to get great results.  Once you've run your model and evaluated the fit, you can come back here and run the next cell again for 100 epochs or longer.  

You can learn more about model.fit [here](https://keras.rstudio.com/reference/fit.html)

In [ ]:
epochs = 5
batch_size = 16 #lower this value if you get a memory error
hist = model.fit(train_X, train_Y, batch_size=batch_size, verbose=True,  epochs=epochs)

### 6. Evaluate the results

Next, we will plot up the true and predicted log(masses).  We have only trained for 5 epochs, so we expect noisy results and catastrophic outliers.  Don't panic -- we will discuss how to fix this in the FAQs. 

In [ ]:
prediction = model.predict(test_X, verbose=0, batch_size=batch_size).flatten()
# Remember when we subtracted off the min in an earlier cell?  In the next line, we're putting it back in!
plt.scatter(test_Y + norm, prediction + norm, c='C0', label='model predictions')
x = np.linspace(np.min(test_Y+norm), np.max(test_Y+norm), 100)
plt.plot(x,x,ls='--', c='C1', label='one-to-one line')
plt.xlabel('True '+r'$\log\left(M_{500c}\right)$')
plt.ylabel('Predicted '+r'$\log\left(M_{500c}\right)$')
plt.title('Results after {:.0f} epochs'.format(epochs))


# Approach 1 outputs
All models, metrics, histories, plots, and plotting scripts are saved under `approach1_outputs`.


In [ ]:
from pathlib import Path
import json
output_dir = Path('approach1_outputs')
output_dir.mkdir(exist_ok=True)


In [ ]:
model.save(output_dir / 'baseline_model.keras')
baseline_prediction = model.predict(test_X, verbose=0, batch_size=batch_size).flatten()
baseline_results = pd.DataFrame([{'r2': r2_score(test_Y, baseline_prediction), 'mae': mean_absolute_error(test_Y, baseline_prediction), 'rmse': np.sqrt(mean_squared_error(test_Y, baseline_prediction)), 'bias': np.mean(test_Y - baseline_prediction)}])
baseline_results.to_csv(output_dir / 'baseline_metrics.csv', index=False)
np.savez(output_dir / 'training_history.npz', loss=np.asarray(hist.history.get('loss', [])))
(output_dir / 'run_metadata.json').write_text(json.dumps({'approach': 1, 'description': 'Original fixed train/test split'}, indent=2))
print(f'Saved baseline model and metrics to {output_dir.resolve()}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(test_Y + norm, baseline_prediction + norm, c='C0')
plot_min = min(np.min(test_Y + norm), np.min(baseline_prediction + norm))
plot_max = max(np.max(test_Y + norm), np.max(baseline_prediction + norm))
axes[0].plot([plot_min, plot_max], [plot_min, plot_max], '--', c='C1')
axes[0].set(xlabel='True log mass', ylabel='Predicted log mass', title='Baseline predictions')
axes[1].plot(hist.history.get('loss', []), label='Training loss')
axes[1].set(xlabel='Epoch', ylabel='MSE', title='Baseline training loss')
axes[1].legend()
fig.tight_layout()
fig.savefig(output_dir / 'baseline_diagnostics.png', dpi=200, bbox_inches='tight')
plt.show()
plt.close(fig)
print('Saved baseline plot')


In [ ]:
plot_script = r'''from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUTPUT_DIR = Path(__file__).resolve().parent
metrics = pd.read_csv(OUTPUT_DIR / 'baseline_metrics.csv')
history = np.load(OUTPUT_DIR / 'training_history.npz')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
metric_names = ['r2', 'mae', 'rmse', 'bias']
axes[0].bar(metric_names, [metrics[name].iloc[0] for name in metric_names], color='C0')
axes[0].set_title('Baseline test metrics')
axes[0].grid(axis='y', alpha=0.25)
axes[1].plot(history['loss'], label='Training loss')
axes[1].set(xlabel='Epoch', ylabel='MSE', title='Baseline training loss')
axes[1].legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'baseline_diagnostics.png', dpi=200, bbox_inches='tight')
plt.close(fig)
print(f'Plots regenerated in {OUTPUT_DIR}')
'''
(output_dir / 'plot_outputs.py').write_text(plot_script)
print(f'Saved plotting script to {(output_dir / "plot_outputs.py").resolve()}')